# 02. 탐색적 데이터 분석 (EDA)

전처리된 AIS 데이터(상위 500척)에 대한 탐색적 분석을 수행한다.

- 선박 유형별 분포
- 시간대별 교통량 패턴
- 속도/침로 분포
- 주요 항로 히트맵

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_parquet('../data/ais_cleaned.parquet')
print(f'{len(df):,} rows, {df["MMSI"].nunique()} vessels')
df.head()

## 2.1 기본 통계

In [ ]:
df[['SOG', 'COG', 'Heading', 'Length', 'Width']].describe()

## 2.2 선박 유형별 분포

In [ ]:
# AIS VesselType 코드 매핑 (주요 유형)
VESSEL_TYPE_MAP = {
    30: 'Fishing', 31: 'Towing', 32: 'Towing (large)',
    33: 'Dredging', 34: 'Diving ops', 35: 'Military',
    36: 'Sailing', 37: 'Pleasure craft',
    40: 'HSC', 50: 'Pilot', 51: 'SAR', 52: 'Tug',
    60: 'Passenger', 70: 'Cargo', 80: 'Tanker'
}

df['VesselTypeGroup'] = df['VesselType'].apply(
    lambda x: VESSEL_TYPE_MAP.get(int(x // 10) * 10, 'Other') if pd.notna(x) else 'Unknown'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 선박 수 기준
vessel_counts = df.groupby('VesselTypeGroup')['MMSI'].nunique().sort_values(ascending=True)
vessel_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Number of Vessels by Type')
axes[0].set_xlabel('Vessel Count')

# 레코드 수 기준
record_counts = df['VesselTypeGroup'].value_counts().sort_values(ascending=True)
record_counts.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Number of Records by Vessel Type')
axes[1].set_xlabel('Record Count')

plt.tight_layout()
plt.savefig('../results/figures/vessel_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 시간대별 교통량 패턴

In [ ]:
df['hour'] = df['BaseDateTime'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 시간대별 레코드 수
hourly = df.groupby('hour').size()
axes[0].bar(hourly.index, hourly.values, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('Record Count')
axes[0].set_title('AIS Records by Hour')
axes[0].set_xticks(range(0, 24))

# 시간대별 활성 선박 수
hourly_vessels = df.groupby('hour')['MMSI'].nunique()
axes[1].bar(hourly_vessels.index, hourly_vessels.values, color='coral', alpha=0.8)
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Active Vessels')
axes[1].set_title('Active Vessels by Hour')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('../results/figures/hourly_traffic.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 속도(SOG) 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 전체 속도 분포
df['SOG'].clip(upper=30).hist(bins=60, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Speed Over Ground (knots)')
axes[0].set_ylabel('Count')
axes[0].set_title('SOG Distribution (clipped at 30 kn)')
axes[0].axvline(df['SOG'].median(), color='red', linestyle='--', label=f'Median: {df["SOG"].median():.1f}')
axes[0].legend()

# 선박 유형별 평균 속도
type_speed = df.groupby('VesselTypeGroup')['SOG'].mean().sort_values(ascending=True)
type_speed.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Mean SOG (knots)')
axes[1].set_title('Average Speed by Vessel Type')

plt.tight_layout()
plt.savefig('../results/figures/speed_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 침로(COG) 분포

In [ ]:
# 이동 중인 선박만 (SOG > 1)
moving = df[df['SOG'] > 1]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='polar')

theta = np.radians(moving['COG'].dropna())
ax.hist(theta, bins=72, color='steelblue', alpha=0.7)
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_title('Course Over Ground Distribution\n(Moving vessels, SOG > 1 kn)', pad=20)

plt.tight_layout()
plt.savefig('../results/figures/cog_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.6 주요 항로 히트맵

In [ ]:
# 이동 중인 선박 위치 히트맵
moving_sample = moving.sample(min(50000, len(moving)), random_state=42)

center = [moving_sample['LAT'].mean(), moving_sample['LON'].mean()]
m = folium.Map(location=center, zoom_start=4, tiles='CartoDB dark_matter')

heat_data = moving_sample[['LAT', 'LON']].values.tolist()
HeatMap(heat_data, radius=8, blur=12, max_zoom=10).add_to(m)

m.save('../results/figures/traffic_heatmap.html')
print('히트맵 저장: results/figures/traffic_heatmap.html')
m

## 2.7 선박별 레코드 수 분포

In [ ]:
vessel_record_counts = df.groupby('MMSI').size()

fig, ax = plt.subplots(figsize=(12, 5))
vessel_record_counts.hist(bins=50, ax=ax, color='steelblue', alpha=0.7)
ax.set_xlabel('Records per Vessel')
ax.set_ylabel('Number of Vessels')
ax.set_title('Distribution of Records per Vessel')
ax.axvline(vessel_record_counts.median(), color='red', linestyle='--',
           label=f'Median: {vessel_record_counts.median():.0f}')
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/records_per_vessel.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'선박당 레코드 수 통계:')
print(vessel_record_counts.describe())